# 02 - Model Training and Selection

This notebook reproduces the validation-driven experiment journey. It starts with simple baselines, compares classical NLP models, and later records controlled tuning and ablation experiments. The test set is intentionally not evaluated here.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys
from datetime import datetime

import joblib
import pandas as pd
from datasets import load_dataset
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import classification_report_dataframe, evaluate_predictions
from src.paths import CHECKPOINTS_DIR, MODELS_DIR, PROCESSED_DATA_DIR, REPORTS_DIR
from src.preprocessing import preprocess_text

pd.set_option("display.max_colwidth", 140)

## Load Prepared Splits

The notebook prefers local prepared parquet files from `01_data_setup.ipynb`. If they are absent, it falls back to the Hugging Face dataset and recreates the minimal text column locally in memory.

In [ ]:
DATASET_NAME = "QCRI/HumAID-all"


def load_split(split: str) -> pd.DataFrame:
    local_path = PROCESSED_DATA_DIR / f"humaid_{split}_minimal.parquet"
    if local_path.exists():
        return pd.read_parquet(local_path)

    dataset = load_dataset(DATASET_NAME, split=split)
    frame = dataset.to_pandas()
    frame["text_minimal"] = frame["tweet_text"].apply(preprocess_text)
    return frame


train_df = load_split("train")
validation_df = load_split("validation")

X_train = train_df["text_minimal"]
y_train = train_df["class_label"]
X_val = validation_df["text_minimal"]
y_val = validation_df["class_label"]
class_names = sorted(y_train.unique())

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Labels:", len(class_names))

## Evaluation Helper

Macro-F1 is the primary selection metric because the dataset is imbalanced and minority operational classes such as missing people and urgent needs should influence model choice.

In [ ]:
experiment_results: list[dict] = []
trained_models: dict[str, object] = {}


def record_result(experiment_id: str, display_name: str, predictions) -> dict:
    result = evaluate_predictions(
        experiment_name=experiment_id,
        y_true=y_val,
        y_pred=predictions,
        labels=class_names,
    )
    result["name"] = display_name
    experiment_results.append(result)
    return result


def fit_pipeline(experiment_id: str, display_name: str, pipeline: Pipeline, X_source=X_train) -> dict:
    pipeline.fit(X_source, y_train)
    predictions = pipeline.predict(X_val)
    trained_models[experiment_id] = pipeline
    return record_result(experiment_id, display_name, predictions)


def show_results() -> pd.DataFrame:
    return (
        pd.DataFrame(experiment_results)
        .sort_values(["macro_f1", "weighted_f1", "accuracy"], ascending=False)
        .reset_index(drop=True)
    )

## E0 - Most-Frequent Dummy Baseline

In [ ]:
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train.to_frame(), y_train)
dummy_predictions = dummy_model.predict(X_val.to_frame())
record_result("E0_dummy_most_frequent", "Most-frequent dummy baseline", dummy_predictions)

## E1 - CountVectorizer Unigram + MultinomialNB

In [ ]:
e1_pipeline = Pipeline(
    [
        ("vectorizer", CountVectorizer(ngram_range=(1, 1), min_df=2, lowercase=False)),
        ("classifier", MultinomialNB(alpha=1.0)),
    ]
)
fit_pipeline("E1_count_unigram_nb", "Count unigram + MultinomialNB", e1_pipeline)

## E2 - CountVectorizer Unigram/Bigram + MultinomialNB

In [ ]:
e2_pipeline = Pipeline(
    [
        ("vectorizer", CountVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False)),
        ("classifier", MultinomialNB(alpha=1.0)),
    ]
)
fit_pipeline("E2_count_1_2gram_nb", "Count unigram/bigram + MultinomialNB", e2_pipeline)

## E3 - TF-IDF Unigram + MultinomialNB

In [ ]:
e3_pipeline = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(ngram_range=(1, 1), min_df=2, lowercase=False, sublinear_tf=True),
        ),
        ("classifier", MultinomialNB(alpha=1.0)),
    ]
)
fit_pipeline("E3_tfidf_unigram_nb", "TF-IDF unigram + MultinomialNB", e3_pipeline)

## E4 - TF-IDF Unigram + Logistic Regression

In [ ]:
e4_pipeline = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(ngram_range=(1, 1), min_df=2, lowercase=False, sublinear_tf=True),
        ),
        (
            "classifier",
            LogisticRegression(C=1.0, max_iter=1000, solver="liblinear", class_weight=None, random_state=42),
        ),
    ]
)
fit_pipeline("E4_tfidf_unigram_lr", "TF-IDF unigram + Logistic Regression", e4_pipeline)

## E5 - Balanced Logistic Regression

In [ ]:
e5_pipeline = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(ngram_range=(1, 1), min_df=2, lowercase=False, sublinear_tf=True),
        ),
        (
            "classifier",
            LogisticRegression(C=1.0, max_iter=1000, solver="liblinear", class_weight="balanced", random_state=42),
        ),
    ]
)
fit_pipeline("E5_tfidf_unigram_lr_balanced", "TF-IDF unigram + balanced Logistic Regression", e5_pipeline)

## E6 - Word Unigram/Bigram Balanced Logistic Regression

In [ ]:
e6_pipeline = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False, sublinear_tf=True),
        ),
        (
            "classifier",
            LogisticRegression(C=1.0, max_iter=1000, solver="liblinear", class_weight="balanced", random_state=42),
        ),
    ]
)
fit_pipeline("E6_tfidf_1_2gram_lr_balanced", "TF-IDF unigram/bigram + balanced Logistic Regression", e6_pipeline)

## E7 - LinearSVC

In [ ]:
e7_pipeline = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False, sublinear_tf=True),
        ),
        ("classifier", LinearSVC(C=1.0, class_weight="balanced", max_iter=5000, random_state=42)),
    ]
)
fit_pipeline("E7_tfidf_1_2gram_linearsvc_balanced", "TF-IDF unigram/bigram + balanced LinearSVC", e7_pipeline)

## E8 - Character TF-IDF LinearSVC

In [ ]:
e8_pipeline = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, lowercase=False, sublinear_tf=True),
        ),
        ("classifier", LinearSVC(C=1.0, class_weight="balanced", max_iter=5000, random_state=42)),
    ]
)
fit_pipeline("E8_char_3_5gram_linearsvc_balanced", "Character TF-IDF + balanced LinearSVC", e8_pipeline)

## E9 - Word + Character Feature Union

In [ ]:
e9_pipeline = Pipeline(
    [
        (
            "features",
            FeatureUnion(
                [
                    ("word_tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=False)),
                    ("char_tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True, lowercase=False)),
                ]
            ),
        ),
        ("classifier", LinearSVC(C=1.0, class_weight="balanced", max_iter=5000, random_state=42)),
    ]
)
fit_pipeline("E9_word_char_tfidf_linearsvc_balanced", "Word + character TF-IDF + balanced LinearSVC", e9_pipeline)

## Baseline Comparison Through E9

Balanced Logistic Regression with word unigrams and bigrams is the strongest model family at this stage. Later cells tune `C` and test controlled raw/preprocessing ablations without touching the test set.

In [ ]:
results_df = show_results()
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
results_df.to_csv(REPORTS_DIR / "validation_results.csv", index=False)
results_df